# MLP HIV Classification with 5-Fold CV

This notebook evaluates the MLP model on the HIV dataset using Morgan fingerprints, stratified 5-fold cross-validation, fold-level validation/test splits, and summary statistics (mean, std, variance).

In [ ]:

# If needed in Colab:
!pip -q install rdkit

import os
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem
RDLogger.DisableLog('rdApp.*')

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import StandardScaler


In [2]:

SEED = 42
N_SPLITS = 5
VAL_SIZE_WITHIN_TEMP = 0.5  # temp set is split equally into val and test
RADIUS = 2
N_BITS = 2048

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)

set_seed(SEED)

def smiles_to_morgan_fp(smiles_list, radius=2, n_bits=2048):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
            fps.append(np.array(fp, dtype=np.float32))
        else:
            fps.append(np.zeros(n_bits, dtype=np.float32))
    return np.array(fps, dtype=np.float32)

def build_model(random_state):
    from sklearn.neural_network import MLPClassifier
    return MLPClassifier(
        hidden_layer_sizes=(100,),
        max_iter=200,
        random_state=random_state
    )

def maybe_scale(X_train, X_val, X_test):
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)
    return X_train, X_val, X_test


def evaluate_fold(model, X_test, y_test):
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_score)
    }, y_score


In [ ]:

df = pd.read_csv("https://raw.githubusercontent.com/McahitKutsal/hivcsv/main/HIV7.csv")
df = df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

X = smiles_to_morgan_fp(df["smiles"], radius=RADIUS, n_bits=N_BITS)
y = df["HIV_active"].astype(int).values

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

fold_results = []
roc_curves = []

for fold, (train_idx, temp_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n===== FOLD {fold} / {N_SPLITS} =====")
    set_seed(SEED + fold)

    X_train_full, y_train_full = X[train_idx], y[train_idx]
    X_temp, y_temp = X[temp_idx], y[temp_idx]

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=VAL_SIZE_WITHIN_TEMP,
        stratify=y_temp,
        random_state=SEED + fold
    )

    X_train, X_val, X_test = maybe_scale(X_train_full, X_val, X_test)

    model = build_model(random_state=SEED + fold)
    model.fit(X_train, y_train_full)

    metrics, y_score = evaluate_fold(model, X_test, y_test)
    fold_results.append({
        "fold": fold,
        "test_accuracy": metrics["accuracy"],
        "test_precision": metrics["precision"],
        "test_recall": metrics["recall"],
        "test_f1": metrics["f1"],
        "test_roc_auc": metrics["roc_auc"]
    })

    fpr, tpr, _ = roc_curve(y_test, y_score)
    roc_curves.append((fold, fpr, tpr, metrics["roc_auc"]))

    print({k: round(v, 4) if isinstance(v, float) else v for k, v in fold_results[-1].items()})

results_df = pd.DataFrame(fold_results)
results_df


In [ ]:

summary_rows = []
for metric in ["test_accuracy", "test_precision", "test_recall", "test_f1", "test_roc_auc"]:
    mean_val = results_df[metric].mean()
    std_val = results_df[metric].std(ddof=1)
    var_val = results_df[metric].var(ddof=1)
    summary_rows.append({
        "Metric": metric.replace("test_", "").upper(),
        "Mean": mean_val,
        "Std": std_val,
        "Variance": var_val,
        "Formatted": f"{mean_val:.4f} ± {std_val:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
summary_df


In [ ]:

plt.figure(figsize=(8, 6))
for fold, fpr, tpr, auc_val in roc_curves:
    plt.plot(fpr, tpr, label=f"Fold {fold} (AUC={auc_val:.3f})")
plt.plot([0, 1], [0, 1], "k--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("MLP ROC Curves Across 5 Folds")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:

results_df.to_csv("MLP_fold_results.csv", index=False)
summary_df.to_csv("MLP_summary_results.csv", index=False)

print("Saved:")
print("- MLP_fold_results.csv")
print("- MLP_summary_results.csv")


In [ ]:
# ================================
# MLP FINAL PIPELINE (TOP-2 + DOCKING)
# ================================

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski, Crippen, QED
from sklearn.model_selection import train_test_split

# Rebuild test SMILES for the last fold
_, test_idx_local = train_test_split(
    temp_idx,
    test_size=VAL_SIZE_WITHIN_TEMP,
    stratify=y_temp,
    random_state=SEED + fold
)

smiles_test = df.iloc[test_idx_local]["smiles"].reset_index(drop=True)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

df_pred = pd.DataFrame({
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10)

def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
        "RotBonds": Lipinski.NumRotatableBonds(mol),
        "QED": QED.qed(mol)
    }

desc_list = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d:
        d.update(row.to_dict())
        desc_list.append(d)

df_desc = pd.DataFrame(desc_list)

df_desc["Lipinski"] = (
    (df_desc["MW"] <= 500) &
    (df_desc["LogP"] <= 5) &
    (df_desc["HBD"] <= 5) &
    (df_desc["HBA"] <= 10)
)

filtered_df = df_desc[df_desc["Lipinski"] == True]

final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2)

print(final_df[["smiles", "y_prob", "QED"]])

final_df["smiles"].to_csv("MLP_docking_input.smi", index=False, header=False)

mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(300,300)
)

display(img)


In [ ]:
# ================================
# MLP FINAL PIPELINE (TOP-2 + DOCKING + FULL SMILES FIX)
# ================================

import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
from rdkit.Chem import Lipinski, Crippen, QED
from sklearn.model_selection import train_test_split

# 🔴 SMILES kesilmesini engelle
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Rebuild test SMILES for the last fold
_, test_idx_local = train_test_split(
    temp_idx,
    test_size=VAL_SIZE_WITHIN_TEMP,
    stratify=y_temp,
    random_state=SEED + fold
)

smiles_test = df.iloc[test_idx_local]["smiles"].reset_index(drop=True)

y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob > 0.5).astype(int)

df_pred = pd.DataFrame({
    "smiles": smiles_test,
    "y_true": y_test,
    "y_pred": y_pred,
    "y_prob": y_prob
})

top_df = df_pred.sort_values(by="y_prob", ascending=False).head(10).reset_index(drop=True)

def compute_desc(sm):
    mol = Chem.MolFromSmiles(sm)
    if mol is None:
        return None
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "TPSA": Descriptors.TPSA(mol),
        "RotBonds": Lipinski.NumRotatableBonds(mol),
        "QED": QED.qed(mol)
    }

desc_list = []
for _, row in top_df.iterrows():
    d = compute_desc(row["smiles"])
    if d:
        d.update(row.to_dict())
        desc_list.append(d)

df_desc = pd.DataFrame(desc_list)

df_desc["Lipinski"] = (
    (df_desc["MW"] <= 500) &
    (df_desc["LogP"] <= 5) &
    (df_desc["HBD"] <= 5) &
    (df_desc["HBA"] <= 10)
)

print("\nMLP TOP 10 CANDIDATES:")
display(df_desc)

# 🔥 FULL SMILES (TOP 10)
print("\nFULL SMILES (TOP 10):")
for i, smi in enumerate(df_desc["smiles"], start=1):
    print(f"{i}. {smi}")

filtered_df = df_desc[df_desc["Lipinski"] == True].copy()

if len(filtered_df) >= 2:
    final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()
else:
    final_df = df_desc.sort_values(by=["QED", "y_prob"], ascending=False).head(2).copy()

print("\nMLP FINAL 2 CANDIDATES:")
display(final_df[["smiles", "y_prob", "QED", "MW", "LogP", "Lipinski"]])

# 🔥 FULL SMILES (FINAL 2)
print("\nFULL SMILES (FINAL 2):")
for i, smi in enumerate(final_df["smiles"], start=1):
    print(f"{i}. {smi}")

final_df["smiles"].to_csv("MLP_docking_input.smi", index=False, header=False)
print("\nSaved: MLP_docking_input.smi")

mols = [Chem.MolFromSmiles(sm) for sm in final_df["smiles"]]

img = Draw.MolsToGridImage(
    mols,
    molsPerRow=2,
    subImgSize=(300, 300),
    legends=[
        f"MLP Mol1\nProb={final_df.iloc[0]['y_prob']:.3f}" if len(final_df) > 0 else "",
        f"MLP Mol2\nProb={final_df.iloc[1]['y_prob']:.3f}" if len(final_df) > 1 else ""
    ]
)

display(img)

In [ ]:
print("\nFINAL 2 MOLECULES:")
print(final_df)

# nicer format if you prefer:
display(final_df)

final_df = filtered_df.sort_values(by=["QED", "y_prob"], ascending=False).head(2)

# fix column order
final_df = final_df[[
    "MW", "LogP", "HBD", "HBA", "TPSA", "RotBonds",
    "QED", "smiles", "fold", "y_true", "y_pred", "y_prob", "Lipinski"
]]

print("\nFINAL 2 MOLECULES:")
display(final_df)